In [ ]:
%pip install -q --no-cache-dir "geemap==0.36.6" folium seaborn xyzservices python-box


In [ ]:
from pathlib import Path
import re
import shutil

import ee
import folium
import geemap.foliumap as geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from google.colab import drive, files

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


In [ ]:
EE_PROJECT = "water-qual-500217"

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized:", EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine authenticated and initialized:", EE_PROJECT)


In [ ]:

BASE = Path("/content")

# This notebook is designed for BE 2562-2568 (AD 2019-2025).
# Hourly strict version: uses survey_time, GSMaP hourly rainfall, and ERA5-Land Hourly.
YEAR_BE_MIN = 2562
YEAR_BE_MAX = 2568
INPUT_FILENAME = f"wq_14_{YEAR_BE_MIN}_{YEAR_BE_MAX}_rs_ready.csv"
IN_CSV = BASE / INPUT_FILENAME

OUT_DIR = BASE / "tabular_data_14/03_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if not IN_CSV.exists():
    print(f"{IN_CSV.name} was not found in /content. Upload the rs_ready CSV now.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV file was uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    IN_CSV = BASE / uploaded_name

print("Using input CSV:", IN_CSV)


In [ ]:

field = pd.read_csv(IN_CSV, encoding="utf-8-sig")
field["survey_date"] = pd.to_datetime(field["survey_date"], errors="coerce").dt.normalize()

if "survey_time" not in field.columns:
    raise KeyError(
        "Missing required column: survey_time. "
        "Hourly strict features need the actual local sampling time."
    )


def normalize_survey_time(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "nat", "none"}:
        return None

    text = text.replace("?.", "").replace("?", "").strip()
    if re.fullmatch(r"\d{1,2}\.\d{2}(\.\d{2})?", text):
        text = text.replace(".", ":")

    # Excel time fraction, e.g. 0.5 = 12:00:00.
    try:
        numeric_value = float(text)
        if 0 <= numeric_value < 1:
            seconds = int(round(numeric_value * 24 * 60 * 60)) % (24 * 60 * 60)
            hh = seconds // 3600
            mm = (seconds % 3600) // 60
            ss = seconds % 60
            return f"{hh:02d}:{mm:02d}:{ss:02d}"
    except ValueError:
        pass

    # Compact HHMM/HMM notation.
    if re.fullmatch(r"\d{3,4}", text):
        compact = text.zfill(4)
        return f"{compact[:2]}:{compact[2:]}:00"

    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.strftime("%H:%M:%S")


field["survey_time_clean"] = field["survey_time"].apply(normalize_survey_time)
field["sample_datetime_local_dt"] = pd.to_datetime(
    field["survey_date"].dt.strftime("%Y-%m-%d") + " " + field["survey_time_clean"].fillna(""),
    errors="coerce",
)

bad_time_rows = field[field[["survey_date", "survey_time_clean", "sample_datetime_local_dt"]].isna().any(axis=1)]
if len(bad_time_rows):
    display(bad_time_rows[["survey_date", "survey_time"]].head(20))
    raise ValueError(f"Found {len(bad_time_rows)} rows with missing/unparseable survey_time.")

sample_datetime_utc_dt = (
    field["sample_datetime_local_dt"]
    .dt.tz_localize("Asia/Bangkok")
    .dt.tz_convert("UTC")
)

# Strict cutoff: use only complete hourly images before the sampling time.
# Example: sample at 09:30 local -> cutoff is 09:00 local; the 09:00-10:00 hour is excluded.
sample_cutoff_utc_dt = sample_datetime_utc_dt.dt.floor("h")
sample_cutoff_local_dt = field["sample_datetime_local_dt"].dt.floor("h")

field["sample_datetime_local"] = field["sample_datetime_local_dt"].dt.strftime("%Y-%m-%dT%H:%M:%S+07:00")
field["sample_datetime_utc"] = sample_datetime_utc_dt.dt.strftime("%Y-%m-%dT%H:%M:%SZ")
field["sample_cutoff_local"] = sample_cutoff_local_dt.dt.strftime("%Y-%m-%dT%H:%M:%S+07:00")
field["sample_cutoff_utc"] = sample_cutoff_utc_dt.dt.strftime("%Y-%m-%dT%H:%M:%SZ")
field["minutes_after_cutoff"] = (
    (field["sample_datetime_local_dt"] - sample_cutoff_local_dt) / pd.Timedelta(minutes=1)
).astype(int)

if "year_be" not in field.columns:
    field["year_be"] = field["survey_date"].dt.year + 543

field["year_be"] = pd.to_numeric(field["year_be"], errors="coerce")
field = field[field["year_be"].between(YEAR_BE_MIN, YEAR_BE_MAX)].copy()
field["year_be"] = field["year_be"].astype(int)
field["year_ad"] = field["year_be"] - 543

if "rs_use_status" in field.columns:
    field = field[field["rs_use_status"].fillna("keep").eq("keep")].copy()

if "station_canonical" not in field.columns:
    field["station_canonical"] = field["station"].astype(str)

if "station_original" not in field.columns:
    field["station_original"] = field["station_canonical"].astype(str)

field["survey_no"] = (
    field["survey_no"]
    .astype(str)
    .str.replace(r"/+", "_", regex=True)
    .str.strip("_")
)

field["sample_id"] = (
    field["station_canonical"].astype(str)
    + "_"
    + field["survey_no"].astype(str).str.replace("/", "_", regex=False)
    + "_"
    + field["survey_date"].dt.strftime("%Y%m%d")
)

required_cols = [
    "sample_id",
    "station_canonical",
    "survey_no",
    "survey_date",
    "survey_time_clean",
    "sample_datetime_utc",
    "sample_cutoff_utc",
    "latitude",
    "longitude",
]
missing_required = [col for col in required_cols if col not in field.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

bad_rows = field[field[required_cols].isna().any(axis=1)].copy()
if len(bad_rows):
    display(bad_rows[required_cols].head(20))
    raise ValueError(f"Found {len(bad_rows)} rows with missing required metadata.")

print("Filtered shape:", field.shape)
print("Date range:", field["survey_date"].min(), "to", field["survey_date"].max())
print("Sample datetime UTC range:", field["sample_datetime_utc"].min(), "to", field["sample_datetime_utc"].max())
print("Rows by BE year:")
display(field.groupby("year_be").size().rename("rows"))
print("Unique stations:", field["station_canonical"].nunique())

field[[
    "sample_id",
    "station_original",
    "station_canonical",
    "survey_date",
    "survey_time_clean",
    "sample_datetime_local",
    "sample_cutoff_local",
    "minutes_after_cutoff",
    "latitude",
    "longitude",
]].head()


In [ ]:

CONTEXT_RUN_TAG = f"ctx_{YEAR_BE_MIN}_{YEAR_BE_MAX}_gsmap_era5h_strict_v1"
CONTEXT_DRIVE_FOLDER = f"rswq_context_features_{CONTEXT_RUN_TAG}"
CONTEXT_CHUNK_PREFIX = f"rswq_context_{CONTEXT_RUN_TAG}_chunk"
CONTEXT_FINAL_NAME = f"rswq_context_features_{CONTEXT_RUN_TAG}.csv"

NODATA = -9999
CHUNK_SIZE = 50

RAIN_BUFFER_M = 10000
WEATHER_BUFFER_M = 15000
LAND_BUFFERS_M = [500, 1000, 5000]
WATER_CONTEXT_BUFFER_M = 500
TERRAIN_BUFFER_M = 500

field["field_date"] = field["survey_date"].dt.strftime("%Y-%m-%d")
field["month"] = field["survey_date"].dt.month.astype(int)
field["day_of_year"] = field["survey_date"].dt.dayofyear.astype(int)
field["doy_sin"] = np.sin(2 * np.pi * field["day_of_year"] / 365.25)
field["doy_cos"] = np.cos(2 * np.pi * field["day_of_year"] / 365.25)

field["season_south_th"] = np.select(
    [
        field["month"].isin([2, 3, 4]),
        field["month"].isin([5, 6, 7, 8, 9, 10]),
        field["month"].isin([11, 12, 1]),
    ],
    ["dry_hot", "sw_monsoon", "ne_monsoon"],
    default="unknown",
)

CONTEXT_META_COLS = [
    "sample_id",
    "station_original",
    "station_canonical",
    "station_survey_id",
    "canonical_station_survey_id",
    "survey_no",
    "field_date",
    "survey_time",
    "survey_time_clean",
    "sample_datetime_local",
    "sample_datetime_utc",
    "sample_cutoff_local",
    "sample_cutoff_utc",
    "minutes_after_cutoff",
    "year_be",
    "year_ad",
    "month",
    "day_of_year",
    "doy_sin",
    "doy_cos",
    "season_south_th",
    "latitude",
    "longitude",
    "water_body",
    "coordinate_waterbody",
    "station_location",
    "coordinate_assignment_status",
]

points_context = field[[c for c in CONTEXT_META_COLS if c in field.columns]].copy()
print("points_context:", points_context.shape)
display(points_context.head())


In [ ]:
def _to_jsonable(value):
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


def pandas_to_ee_points(df, lon_col="longitude", lat_col="latitude"):
    features = []
    for _, row in df.iterrows():
        geom = ee.Geometry.Point([float(row[lon_col]), float(row[lat_col])])
        props = {col: _to_jsonable(row[col]) for col in df.columns}
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)


ee_context_points = pandas_to_ee_points(points_context)
print("EE context points:", ee_context_points.size().getInfo())

In [ ]:

GSMAP_ASSET = "JAXA/GPM_L3/GSMaP/v8/operational"
GSMAP_RAIN_BAND = "hourlyPrecipRateGC"  # gauge-adjusted hourly rain rate, mm/hr
GSMAP = ee.ImageCollection(GSMAP_ASSET).select(GSMAP_RAIN_BAND)

ERA5H = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
WORLDCOVER = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
JRC_WATER = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
SRTM = ee.Image("USGS/SRTMGL1_003").select("elevation")
SLOPE = ee.Terrain.slope(SRTM).rename("slope")

# --- Strict hourly prior windows ------------------------------------------
# sample_cutoff_utc = previous complete hour before sample time.
# Example: sample 09:30 Asia/Bangkok -> cutoff 09:00 local / 02:00 UTC.
# All hourly windows are [cutoff - N hours, cutoff), so no image can cross the sample time.
HOURLY_RAIN_WINDOWS_H = [1, 3, 6, 12, 24, 72, 168, 336, 720]
HOURLY_WEATHER_WINDOWS_H = [1, 3, 6, 12, 24, 72, 168]

GSMAP_RAIN_THRESHOLD_MM_H = 1.0
HOURS_SINCE_RAIN_LOOKBACK = 720


def suffix_for_hours(hours):
    return f"{int(hours)}h"


def prior_hour_window(cutoff_time, hours):
    start = cutoff_time.advance(-int(hours), "hour")
    end = cutoff_time
    return start, end


def reduce_mean_dict(image, geom, scale):
    return image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=scale,
        maxPixels=1e8,
        bestEffort=True,
        tileScale=4,
    )


def make_gsmap_rain_image(cutoff_time):
    bands = []

    for hours in HOURLY_RAIN_WINDOWS_H:
        start, end = prior_hour_window(cutoff_time, hours)
        suffix = suffix_for_hours(hours)
        coll = GSMAP.filterDate(start, end)

        rain_sum = coll.sum().rename(f"rain_gsmap_prior_{suffix}_mm")
        rain_max = coll.max().rename(f"rain_gsmap_max_prior_{suffix}_mm_h")
        rainy_hours = (
            coll
            .map(lambda img: img.gt(GSMAP_RAIN_THRESHOLD_MM_H).rename("rainy_hour"))
            .sum()
            .rename(f"rainy_hours_gsmap_prior_{suffix}")
        )

        bands.extend([rain_sum, rain_max, rainy_hours])

    lookback_start = cutoff_time.advance(-HOURS_SINCE_RAIN_LOOKBACK, "hour")
    lookback_coll = GSMAP.filterDate(lookback_start, cutoff_time)

    def tag_offset(img):
        img_time = ee.Date(img.get("system:time_start"))
        offset = cutoff_time.difference(img_time, "hour")
        is_rainy = img.gt(GSMAP_RAIN_THRESHOLD_MM_H)
        offset_img = ee.Image.constant(offset).toFloat()
        return offset_img.where(is_rainy.Not(), HOURS_SINCE_RAIN_LOOKBACK + 1).rename("candidate_offset")

    hours_since_rain_img = (
        lookback_coll
        .map(tag_offset)
        .min()
        .rename("hours_since_last_rain_gsmap_mean")
    )
    bands.append(hours_since_rain_img)

    return ee.Image.cat(bands)


def prep_era5h(img):
    air_temp_c = img.select("temperature_2m").subtract(273.15).rename("air_temp_C")
    dewpoint_c = img.select("dewpoint_temperature_2m").subtract(273.15).rename("dewpoint_C")
    u = img.select("u_component_of_wind_10m")
    v = img.select("v_component_of_wind_10m")
    wind = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed_m_s")

    # ERA5-Land Hourly accumulated variables are per forecast step in the GEE catalog.
    runoff_mm = img.select("surface_runoff").multiply(1000).rename("surface_runoff_mm")
    solar_mj = img.select("surface_solar_radiation_downwards").divide(1e6).rename("solar_MJ_m2")

    return ee.Image.cat([air_temp_c, dewpoint_c, wind, runoff_mm, solar_mj]).copyProperties(
        img,
        ["system:time_start"],
    )


def make_era5h_weather_image(cutoff_time):
    bands = []
    for hours in HOURLY_WEATHER_WINDOWS_H:
        start, end = prior_hour_window(cutoff_time, hours)
        suffix = suffix_for_hours(hours)
        coll = ERA5H.filterDate(start, end).map(prep_era5h)
        mean_img = coll.mean()
        max_img = coll.max()
        sum_img = coll.sum()

        bands.extend([
            mean_img.select("air_temp_C").rename(f"era5h_air_temp_prior_{suffix}_C"),
            mean_img.select("dewpoint_C").rename(f"era5h_dewpoint_prior_{suffix}_C"),
            mean_img.select("wind_speed_m_s").rename(f"era5h_wind_mean_prior_{suffix}_m_s"),
            max_img.select("wind_speed_m_s").rename(f"era5h_wind_max_prior_{suffix}_m_s"),
            sum_img.select("surface_runoff_mm").rename(f"era5h_runoff_prior_{suffix}_mm"),
            sum_img.select("solar_MJ_m2").rename(f"era5h_solar_prior_{suffix}_MJ_m2"),
        ])

    return ee.Image.cat(bands)


WORLDCOVER_CLASSES = {
    "tree": 10,
    "shrub": 20,
    "grass": 30,
    "crop": 40,
    "built": 50,
    "bare": 60,
    "water": 80,
    "wetland": 90,
    "mangrove": 95,
}


def make_worldcover_fraction_image(buffer_m):
    suffix = f"{int(buffer_m)}m"
    bands = []
    for name, class_value in WORLDCOVER_CLASSES.items():
        bands.append(WORLDCOVER.eq(class_value).rename(f"wc_{name}_frac_{suffix}"))
    return ee.Image.cat(bands)


JRC_CONTEXT_IMAGE = ee.Image.cat([
    JRC_WATER.select("occurrence").rename("jrc_occurrence_mean_500m"),
    JRC_WATER.select("seasonality").rename("jrc_seasonality_mean_500m"),
    JRC_WATER.select("recurrence").rename("jrc_recurrence_mean_500m"),
])

TERRAIN_CONTEXT_IMAGE = ee.Image.cat([
    SRTM.rename("srtm_elevation_mean_500m"),
    SLOPE.rename("srtm_slope_mean_500m"),
])


def extract_context_features_fast(feature):
    cutoff_time = ee.Date(feature.get("sample_cutoff_utc"))
    geom = feature.geometry()

    rain_stats = reduce_mean_dict(make_gsmap_rain_image(cutoff_time), geom.buffer(RAIN_BUFFER_M), 11132)
    weather_stats = reduce_mean_dict(make_era5h_weather_image(cutoff_time), geom.buffer(WEATHER_BUFFER_M), 11132)

    out = feature.set(rain_stats).set(weather_stats)

    for buffer_m in LAND_BUFFERS_M:
        land_stats = reduce_mean_dict(
            make_worldcover_fraction_image(buffer_m),
            geom.buffer(buffer_m),
            10,
        )
        out = out.set(land_stats)

    water_stats = reduce_mean_dict(JRC_CONTEXT_IMAGE, geom.buffer(WATER_CONTEXT_BUFFER_M), 30)
    terrain_stats = reduce_mean_dict(TERRAIN_CONTEXT_IMAGE, geom.buffer(TERRAIN_BUFFER_M), 30)

    return out.set(water_stats).set(terrain_stats)


context_fc = ee_context_points.map(extract_context_features_fast)
print("context_fc ready:", context_fc.size().getInfo())
print("GSMaP asset:", GSMAP_ASSET, "band:", GSMAP_RAIN_BAND)
print("Hourly cutoff property: sample_cutoff_utc")


In [ ]:
# Test only 1 row interactively. For all rows, use Drive export.
test_context = geemap.ee_to_df(context_fc.limit(1))
display(test_context.T)

In [ ]:

CONTEXT_FEATURE_COLS = []

for hours in HOURLY_RAIN_WINDOWS_H:
    suffix = suffix_for_hours(hours)
    CONTEXT_FEATURE_COLS.extend([
        f"rain_gsmap_prior_{suffix}_mm",
        f"rain_gsmap_max_prior_{suffix}_mm_h",
        f"rainy_hours_gsmap_prior_{suffix}",
    ])

CONTEXT_FEATURE_COLS.append("hours_since_last_rain_gsmap_mean")

for hours in HOURLY_WEATHER_WINDOWS_H:
    suffix = suffix_for_hours(hours)
    CONTEXT_FEATURE_COLS.extend([
        f"era5h_air_temp_prior_{suffix}_C",
        f"era5h_dewpoint_prior_{suffix}_C",
        f"era5h_wind_mean_prior_{suffix}_m_s",
        f"era5h_wind_max_prior_{suffix}_m_s",
        f"era5h_runoff_prior_{suffix}_mm",
        f"era5h_solar_prior_{suffix}_MJ_m2",
    ])

CONTEXT_FEATURE_COLS.extend([
    "jrc_occurrence_mean_500m",
    "jrc_seasonality_mean_500m",
    "jrc_recurrence_mean_500m",
    "srtm_elevation_mean_500m",
    "srtm_slope_mean_500m",
])

for buffer_m in LAND_BUFFERS_M:
    suffix = f"{int(buffer_m)}m"
    for land_name in WORLDCOVER_CLASSES.keys():
        CONTEXT_FEATURE_COLS.append(f"wc_{land_name}_frac_{suffix}")

CONTEXT_EXPORT_COLS = [c for c in CONTEXT_META_COLS if c in points_context.columns] + CONTEXT_FEATURE_COLS
CONTEXT_STRING_COLS = [
    c for c in [
        "sample_id",
        "station_original",
        "station_canonical",
        "station_survey_id",
        "canonical_station_survey_id",
        "survey_no",
        "field_date",
        "survey_time",
        "survey_time_clean",
        "sample_datetime_local",
        "sample_datetime_utc",
        "sample_cutoff_local",
        "sample_cutoff_utc",
        "season_south_th",
        "water_body",
        "coordinate_waterbody",
        "station_location",
        "coordinate_assignment_status",
    ]
    if c in CONTEXT_EXPORT_COLS
]
CONTEXT_NUMERIC_COLS = [c for c in CONTEXT_EXPORT_COLS if c not in CONTEXT_STRING_COLS]


def _ee_values_with_defaults(feature, cols, default_value):
    col_list = ee.List(cols)

    def value_or_default(col):
        col = ee.String(col)
        value = feature.get(col)
        return ee.Algorithms.If(
            ee.Algorithms.IsEqual(value, None),
            default_value,
            value,
        )

    return ee.Dictionary.fromLists(col_list, col_list.map(value_or_default))


def ensure_context_schema(feature):
    string_values = _ee_values_with_defaults(feature, CONTEXT_STRING_COLS, "")
    numeric_values = _ee_values_with_defaults(feature, CONTEXT_NUMERIC_COLS, NODATA)
    return feature.set(string_values).set(numeric_values)


context_fc_export = context_fc.map(ensure_context_schema).select(CONTEXT_EXPORT_COLS)
print("Export columns:", len(CONTEXT_EXPORT_COLS))
print("Feature columns:", len(CONTEXT_FEATURE_COLS))


In [ ]:
field_chunks = points_context[["sample_id"]].copy()
field_chunks["chunk_id"] = np.arange(len(field_chunks)) // CHUNK_SIZE
points_context_for_chunks = points_context.merge(field_chunks, on="sample_id", how="left")

chunk_plan = (
    points_context_for_chunks
    .groupby("chunk_id")
    .agg(rows=("sample_id", "count"))
    .reset_index()
)
display(chunk_plan)

# Set True only when you are ready to start Earth Engine Drive exports.
START_CONTEXT_EXPORT = True

context_tasks = []
if START_CONTEXT_EXPORT:
    for chunk_id, chunk_df in points_context_for_chunks.groupby("chunk_id"):
        chunk_id_int = int(chunk_id)
        ee_chunk_points = pandas_to_ee_points(chunk_df.drop(columns=["chunk_id"]))
        ee_chunk_context = (
            ee_chunk_points
            .map(extract_context_features_fast)
            .map(ensure_context_schema)
            .select(CONTEXT_EXPORT_COLS)
        )

        task = ee.batch.Export.table.toDrive(
            collection=ee_chunk_context,
            description=f"{CONTEXT_CHUNK_PREFIX}_{chunk_id_int:02d}",
            folder=CONTEXT_DRIVE_FOLDER,
            fileNamePrefix=f"{CONTEXT_CHUNK_PREFIX}_{chunk_id_int:02d}",
            fileFormat="CSV",
            selectors=CONTEXT_EXPORT_COLS,
        )
        task.start()
        context_tasks.append({"chunk_id": chunk_id_int, "rows": len(chunk_df), "task_id": task.id})
        print(f"Started context chunk {chunk_id_int:02d}: rows={len(chunk_df)}, task_id={task.id}")
else:
    print("START_CONTEXT_EXPORT is False. Review the test output first, then set it to True.")

In [ ]:
def list_context_tasks():
    rows = []
    for task in ee.batch.Task.list():
        cfg = task.config or {}
        desc = cfg.get("description", "")
        if str(desc).startswith(CONTEXT_CHUNK_PREFIX):
            status = task.status()
            rows.append({
                "id": task.id,
                "description": desc,
                "state": status.get("state"),
                "error_message": status.get("error_message"),
            })
    return pd.DataFrame(rows)


display(list_context_tasks())

In [ ]:
drive.mount("/content/drive", force_remount=False)

chunk_dir = Path("/content/drive/MyDrive") / CONTEXT_DRIVE_FOLDER
chunk_dir.mkdir(parents=True, exist_ok=True)

raw_chunk_files = sorted(chunk_dir.glob(f"{CONTEXT_CHUNK_PREFIX}_*.csv"))
print("chunk_dir:", chunk_dir)
print("raw chunk files found:", len(raw_chunk_files))
for file_path in raw_chunk_files:
    print(file_path.name, file_path.stat().st_size)

chunk_file_pattern = re.compile(rf"^{re.escape(CONTEXT_CHUNK_PREFIX)}_(\d+).*\.csv$")
chunk_file_rows = []
for file_path in raw_chunk_files:
    match = chunk_file_pattern.match(file_path.name)
    if match:
        chunk_file_rows.append({
            "chunk_id": int(match.group(1)),
            "path": file_path,
            "name": file_path.name,
            "size_bytes": file_path.stat().st_size,
            "modified_time": file_path.stat().st_mtime,
        })

chunk_files_df = pd.DataFrame(chunk_file_rows)
display(chunk_files_df.sort_values(["chunk_id", "modified_time"]) if len(chunk_files_df) else chunk_files_df)

expected_chunks = set(chunk_plan["chunk_id"].astype(int))
found_chunks = set(chunk_files_df["chunk_id"].astype(int)) if len(chunk_files_df) else set()
missing_chunks = sorted(expected_chunks - found_chunks)
if missing_chunks:
    print("Missing chunk CSV files:", missing_chunks)
    display(list_context_tasks())
    raise RuntimeError("Some context chunk CSV files are missing. Wait for tasks to complete or retry missing chunks.")

latest_chunk_files = (
    chunk_files_df
    .sort_values("modified_time")
    .groupby("chunk_id", as_index=False)
    .tail(1)
    .sort_values("chunk_id")
)

chunk_dfs = []
for _, row in latest_chunk_files.iterrows():
    temp = pd.read_csv(row["path"])
    temp["source_context_chunk_file"] = row["name"]
    chunk_dfs.append(temp)

context_features = pd.concat(chunk_dfs, ignore_index=True)

for col in CONTEXT_NUMERIC_COLS:
    if col in context_features.columns:
        context_features[col] = pd.to_numeric(context_features[col], errors="coerce").replace(NODATA, np.nan)

for col in CONTEXT_STRING_COLS:
    if col in context_features.columns:
        context_features[col] = context_features[col].replace("", np.nan)

field_order = points_context[["sample_id"]].copy()
field_order["_field_order"] = np.arange(len(field_order))

context_features = (
    context_features
    .drop_duplicates("sample_id", keep="last")
    .merge(field_order, on="sample_id", how="left")
    .sort_values("_field_order")
    .drop(columns=["_field_order"])
    .reset_index(drop=True)
)

print("merged context_features shape:", context_features.shape)
print("expected rows:", len(points_context))
print("row count ok:", len(context_features) == len(points_context))
display(context_features.head())

In [ ]:
import re

context_selected = context_features[
    [c for c in CONTEXT_EXPORT_COLS if c in context_features.columns]
].copy()

# Safety check: context file must not include water-quality target columns.
target_prefix_re = re.compile(
    r"^(turbidity|ss|ts|tds|do|bod|tcb|fcb|tp|no3|no2|nh3|wqi|ph|conductivity|salinity|temp_water)(_|$)",
    flags=re.IGNORECASE,
)

target_suffix_re = re.compile(
    r"(_clean|_detection_limit|_value_type)$",
    flags=re.IGNORECASE,
)

leaked_cols = [
    col for col in context_selected.columns
    if target_prefix_re.search(col) or target_suffix_re.search(col)
]

assert not leaked_cols, (
    f"Target or field-value columns leaked into context export: {leaked_cols}"
)

missing_rate = (
    context_selected
    .replace(NODATA, np.nan)
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
)
display(missing_rate.head(30))

numeric_context_cols = [c for c in CONTEXT_FEATURE_COLS if c in context_selected.columns]
display(context_selected[numeric_context_cols].describe().T)

In [ ]:

plot_cols = [
    "rain_gsmap_prior_24h_mm",
    "rain_gsmap_prior_168h_mm",
    "rain_gsmap_prior_720h_mm",
    "era5h_air_temp_prior_24h_C",
    "era5h_wind_mean_prior_24h_m_s",
    "era5h_runoff_prior_24h_mm",
    "wc_built_frac_1000m",
    "wc_crop_frac_1000m",
    "wc_water_frac_1000m",
    "jrc_occurrence_mean_500m",
    "srtm_elevation_mean_500m",
]
plot_cols = [c for c in plot_cols if c in context_selected.columns]

ncols = 3
nrows = int(np.ceil(len(plot_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, plot_cols):
    sns.histplot(context_selected[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)

for ax in axes[len(plot_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

box_cols = [
    "rain_gsmap_prior_24h_mm",
    "rain_gsmap_prior_168h_mm",
    "era5h_air_temp_prior_24h_C",
    "era5h_wind_mean_prior_24h_m_s",
    "wc_built_frac_1000m",
    "wc_crop_frac_1000m",
]
box_cols = [c for c in box_cols if c in context_selected.columns]

fig, axes = plt.subplots(len(box_cols), 1, figsize=(10, 3.2 * len(box_cols)))
if len(box_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, box_cols):
    sns.boxplot(data=context_selected, x="season_south_th", y=col, ax=ax)
    sns.stripplot(data=context_selected, x="season_south_th", y=col, color="black", size=2, alpha=0.35, ax=ax)
    ax.set_title(f"{col} by season")

plt.tight_layout()
plt.show()


In [ ]:
corr_cols = [
    c for c in numeric_context_cols
    if context_selected[c].notna().sum() > 50 and context_selected[c].nunique(dropna=True) > 3
]
corr = context_selected[corr_cols].corr(method="spearman")

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="vlag", center=0, square=False)
plt.title("Spearman correlation among context features")
plt.tight_layout()
plt.show()

In [ ]:
final_context_out = OUT_DIR / CONTEXT_FINAL_NAME
context_selected.to_csv(final_context_out, index=False, encoding="utf-8-sig")
print("Saved final context CSV:", final_context_out)
print("Shape:", context_selected.shape)
display(context_selected.head())

files.download(str(final_context_out))